In [ ]:
from data_utils import *
from training import *
from unlearning import *
from MIA import *
import random
import os

In [ ]:
NUM_CLIENTS = 10 
UNLEARN_CLIENTS = [i for i in range(NUM_CLIENTS)] # list of clients you want to unlearn     
TRAIN_PROP = 0.8
TRAIN_PATH = "train_data.pkl"
TEST_PATH = "test_data.pkl"
RAW_TRAIN_PATH = "raw_train_data.pkl"

# control variation in client size
ALPHA_DIRICHLET = 5 # use for Dirichlet splitting
LENIENCY = 0.1
MIN_CLIENT_SIZE = 1000

ROUNDS = 10 # number of non-federated learning rounds
LR = 0.1 # learning rate of non-federated learning
FL_ROUNDS = 10 # number of global learning rounds in federated learning
FL_LR = 0.1 # learning rate of feederated learning

ALPHA = 0.1 # update rate of the client summary
DAMPING = 0.05 # damping factor for the hessian
ETA = None # unlearning stepsize, calculated by program if None
LAMBDAS = None # ideally, this would be a list of parameters tuned for each client

SEED = 0

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# create a folder to store results
RESULT_DIR = None
if RESULT_DIR is None:
    PATH = Path(f"{ALPHA_DIRICHLET}-{ALPHA}-{DAMPING}-{FL_LR}-{NUM_CLIENTS}-{SEED}-basic")
PATH.mkdir(parents=True, exist_ok=True)


In [ ]:
from federated import *


## 1 - Partition the data into clients and create train, test and validation loader

### 1a - Load global train and test loader  

In [ ]:
# Load global dataset
g = torch.Generator()
g.manual_seed(SEED)

train_dataset, train_meta = load_pkl_as_dataset(TRAIN_PATH)
test_dataset, test_meta = load_pkl_as_dataset(TEST_PATH)
input_size = train_meta["input_size"]
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, generator=g)


### 1b - Divide the data over the different clients

#### Option 1: divide clients by Dirichlet paritioning

In [ ]:
y = np.array(train_dataset.y)
client_indices = dirichlet_partitions(y, num_clients=NUM_CLIENTS, alpha=ALPHA_DIRICHLET, seed=SEED)
client_train_idx, client_val_idx = client_train_val_split(
    client_indices,
    train_prop=TRAIN_PROP,
    seed=SEED,
    labels=y,
)
with open(PATH / "partitions.pkl", "wb") as f:
    pickle.dump({"client_train_idx": client_train_idx, "client_val_idx": client_val_idx}, f)


#### Option 2: divide clients by maximizing variance

In [ ]:
# bpi is too skewed for a company split, so use the clustered split
client_train_idx, client_val_idx, split_audit, kept_features = save_default_partitions(
    processed_train_path=TRAIN_PATH,
    raw_train_path=RAW_TRAIN_PATH,
    output_dir=PATH,
    num_clients=NUM_CLIENTS,
    seed=SEED,
    leniency=LENIENCY,
    min_client_size=MIN_CLIENT_SIZE,
    train_prop=TRAIN_PROP,
)
print(split_audit["min_client_size"], split_audit["blocked_rate_min"], split_audit["blocked_rate_max"])


#### Use pre-existing partition

In [ ]:
# USE IF YOU ALREADY HAVE PARTITIONS
with open(PATH / "partitions.pkl", "rb") as f:
    partitions = pickle.load(f)    
    
client_train_idx = partitions['client_train_idx']
client_val_idx = partitions['client_val_idx']

### 1c - Create train and validation data loaders for clients and global

In [ ]:
# Get dataloaders for the clients and the global dataloaders
client_train_loaders, client_val_loaders = make_federated_loaders(
    train_dataset, client_train_idx, client_val_idx, batch_size=32,
    seed=SEED
)

global_val_loader, global_val_idx = make_global_loader_from_clients(
    train_dataset, client_val_idx, batch_size=32, seed=SEED
)

global_train_loader, global_train_idx = make_global_loader_from_clients(
    train_dataset, client_train_idx, batch_size=32, seed=SEED, shuffle=True
)


## 2 - Train the models

### 2a - Non-federated, using all training data

In [ ]:
non_fd_model = Net(input_size=input_size)
train_one_model(non_fd_model, global_train_loader, DEVICE, epochs=ROUNDS, lr=LR)
non_fd_params = non_fd_model.state_dict()

torch.save(non_fd_params, PATH / "non_fd_params.pt")

non_fd_val_results = create_val_results_dict(non_fd_model, client_val_loaders, DEVICE)
non_fd_test_results = create_test_results_dict(non_fd_model, test_loader, DEVICE)


with open(PATH / "non_fd_val_results", "wb") as f:
    pickle.dump(non_fd_val_results, f)

with open(PATH / "non_fd_test_results", "wb") as f:
    pickle.dump(non_fd_test_results, f)

### 2b - Federated, using all training data

In [ ]:
og_model, og_summaries, og_hessian, og_val_results, _, _ , _= \
    simulate_federated_learning(
        client_train_loaders,
        client_val_loaders,
        DEVICE,
        max_rounds=FL_ROUNDS,
        lr=FL_LR,
        alpha=ALPHA,
        )

In [ ]:
with open(PATH / "og_val_results", "wb") as f:
    pickle.dump(og_val_results, f)

In [ ]:
og_test_results = create_test_results_dict(og_model, test_loader, DEVICE)

with open(PATH / 'og_test_results', "wb") as f:
    pickle.dump(og_test_results, f)

In [ ]:
og_params = og_model.state_dict()
torch.save(og_params, PATH / "og_params.pt")
torch.save(og_summaries, PATH / "og_summaries")
torch.save(og_hessian, PATH / "og_hessian")

### 2c - Federated, using only remaining clients

In [ ]:
for client in UNLEARN_CLIENTS:
    retr_model, retr_summaries, retr_hessian, retr_train_val_results, _, _, _=         simulate_federated_learning(
            client_train_loaders[:client]+client_train_loaders[client+1:],
            client_val_loaders[:client]+client_val_loaders[client+1:],
            DEVICE,
            max_rounds=FL_ROUNDS,
            lr=FL_LR,
            alpha=ALPHA,
            )

    retr_params = retr_model.state_dict()

    client_path = PATH / f"{client}"
    client_path.mkdir(parents=True, exist_ok=True)

    torch.save(retr_params, client_path / "retr_params.pt")

    retr_val_results = create_val_results_dict(retr_model, client_val_loaders, DEVICE)
    with open(client_path / "retr_val_results", "wb") as f:
        pickle.dump(retr_val_results, f)

    retr_test_results = create_test_results_dict(retr_model, test_loader, DEVICE)

    with open(client_path / 'retr_test_results', "wb") as f:
        pickle.dump(retr_test_results, f)


###  2d - Unlearned model

In [ ]:
og_params = torch.load(PATH / "og_params.pt", weights_only=True)
og_summaries = torch.load(PATH / "og_summaries")
og_hessian = torch.load(PATH / "og_hessian")

In [ ]:
for client in UNLEARN_CLIENTS:
    unl_params = unlearn(og_params, og_hessian, og_summaries, client, NUM_CLIENTS, ETA, DAMPING)
    client_path = PATH / f"{client}"
    client_path.mkdir(parents=True, exist_ok=True)
    torch.save(unl_params, client_path / "unl_params.pt")

    unl_model = Net(input_size=input_size)
    unl_model.load_state_dict(unl_params)
    unl_val_results = create_val_results_dict(unl_model, client_val_loaders, DEVICE)
    unl_test_results = create_test_results_dict(unl_model, test_loader, DEVICE)

    with open(PATH / f"{client}" / "unl_val_results", "wb") as f:
        pickle.dump(unl_val_results, f)

    with open(PATH / f"{client}" / "unl_test_results", "wb") as f:
        pickle.dump(unl_test_results, f)


### 2e - Finetuned unlearned model

In [ ]:
for client in UNLEARN_CLIENTS:
    unl_params = torch.load(PATH / f"{client}" / "unl_params.pt", weights_only=True)

    ft_model, ft_summaries, ft_hessian, ft_train_val_results, _, _, _ = simulate_federated_learning(
    client_train_loaders[:client] + client_train_loaders[client+1:],
    client_val_loaders[:client] + client_val_loaders[client+1:],
    DEVICE,
    max_rounds=2,
    lr=FL_LR,
    alpha=ALPHA,
    pretrained=unl_params,
    )

    torch.save(ft_model.state_dict(), PATH /  f"{client}" / "ft_params.pt")

    ft_val_results = create_val_results_dict(ft_model, client_val_loaders, DEVICE)
    ft_test_results = create_test_results_dict(ft_model, test_loader, DEVICE)
    
    with open(PATH / f"{client}" / "ft_val_results", "wb") as f:
        pickle.dump(ft_val_results, f)

    with open(PATH / f"{client}" / "ft_test_results", "wb") as f:
        pickle.dump(ft_test_results, f)


## 3 - Create the membership inference attack

### 3a - Create the shadow model

In [ ]:
shadow_model, shadow_summaries, shadow_hessian, shadow_val_results, _, _ ,_ = \
    simulate_federated_learning(
        client_val_loaders,
        client_train_loaders,
        DEVICE,
        max_rounds=FL_ROUNDS,
        lr=FL_LR,
        alpha=ALPHA,
    )
        
shadow_params = shadow_model.state_dict()
torch.save(shadow_params, PATH / "shadow_params.pt")

with open(PATH / "shadow_val_results", "wb") as f:
    pickle.dump(shadow_val_results, f)

shadow_test_results = create_test_results_dict(shadow_model, test_loader, DEVICE)
with open(PATH / "shadow_test_results", "wb") as f:
    pickle.dump(shadow_test_results, f)

### 3b - Train the MIA

In [ ]:
shadow_params = torch.load(PATH / "shadow_params.pt", weights_only=True)
shadow_model = Net(input_size=input_size)
shadow_model.load_state_dict(shadow_params)

shadow_member_idx = list(global_val_idx)
shadow_non_member_idx = list(range(len(test_dataset)))
random.shuffle(shadow_member_idx)
random.shuffle(shadow_non_member_idx)
shadow_sample_size = min(len(shadow_member_idx), len(shadow_non_member_idx))
shadow_non_member_idx = shadow_non_member_idx[:shadow_sample_size]

X_member, y_member = train_dataset[shadow_member_idx[:shadow_sample_size]]
X_non_member, y_non_member = test_dataset[shadow_non_member_idx]

shadow_members = DataLoader(TabularDataset(X_member, y_member), batch_size=32, shuffle=True)
shadow_non_members = DataLoader(TabularDataset(X_non_member, y_non_member), batch_size=32, shuffle=True)

large_attack_model, large_attack_metrics = train_MIA(
    shadow_model,
    shadow_members,
    shadow_non_members,
    DEVICE,
    large=True,
)
small_attack_model, small_attack_metrics = train_MIA(
    shadow_model,
    shadow_members,
    shadow_non_members,
    DEVICE,
    large=False,
)

large_attack_params = large_attack_model.state_dict()
small_attack_params = small_attack_model.state_dict()

torch.save(large_attack_params, PATH / "large_attack_params.pt")
torch.save(small_attack_params, PATH / "small_attack_params.pt")
print(small_attack_metrics)
print(large_attack_metrics)


## 4 - Load the most recent models

In [ ]:
def load_net(fname):
    m = Net(input_size=input_size)
    p = torch.load(PATH / fname, weights_only=True)
    m.load_state_dict(p)
    return p, m

non_fd_params, non_fd_model = load_net("non_fd_params.pt")
og_params, og_model = load_net("og_params.pt")
shadow_params, shadow_model = load_net("shadow_params.pt")

large_attack_model = MIA_large(input_size=5)
small_attack_model = MIA_small(input_size=5)

large_attack_model.load_state_dict(torch.load(PATH / "large_attack_params.pt", weights_only=True))
small_attack_model.load_state_dict(torch.load(PATH / "small_attack_params.pt", weights_only=True))

In [ ]:
eval_test_idx = [idx for idx in range(len(test_dataset)) if idx not in set(shadow_non_member_idx)]

def create_test_loader(length):
    sample_idx = eval_test_idx.copy()
    random.shuffle(sample_idx)
    X_test, y_test = test_dataset[sample_idx[:length]]
    return DataLoader(TabularDataset(X_test, y_test), batch_size=32, shuffle=True)


In [ ]:
def create_remaining_loader(length, client_train_idx, client):
    
    a = list(np.hstack(client_train_idx[:client] + client_train_idx[client+1:]))
    random.shuffle(a)
    X_rem, y_rem = train_dataset[a[:length]]
    sample_remaining_loader = DataLoader(TabularDataset(X_rem,  y_rem),  batch_size=32, shuffle=True)

    return sample_remaining_loader


## 5 - Test the unlearning performance

### 5a - MIA results

In [ ]:
# TEST ON SMALL MODEL

MIA_small_results = {}

for client in UNLEARN_CLIENTS:
    retr_params = torch.load(PATH / f"{client}" / "retr_params.pt", weights_only=True)
    unl_params = torch.load(PATH / f"{client}" / "unl_params.pt", weights_only=True)
    ft_params = torch.load(PATH / f"{client}" / "ft_params.pt", weights_only=True)

    forgotten_loader = client_train_loaders[client]
    sample_size = len(forgotten_loader.dataset)
    test_non_member_loader = create_test_loader(sample_size)
    remaining_member_loader = create_remaining_loader(sample_size, client_train_idx, client)

    original_metrics = membership_inference_attack(
        og_params,
        test_non_member_loader,
        forgotten_loader,
        small_attack_model,
        DEVICE,
        focus_loader=forgotten_loader,
    )
    retrain_metrics = membership_inference_attack(
        retr_params,
        forgotten_loader,
        remaining_member_loader,
        small_attack_model,
        DEVICE,
        focus_loader=forgotten_loader,
    )
    unlearn_metrics = membership_inference_attack(
        unl_params,
        forgotten_loader,
        remaining_member_loader,
        small_attack_model,
        DEVICE,
        focus_loader=forgotten_loader,
    )
    finetune_metrics = membership_inference_attack(
        ft_params,
        forgotten_loader,
        remaining_member_loader,
        small_attack_model,
        DEVICE,
        focus_loader=forgotten_loader,
    )

    MIA_small_results[client] = {
        "original": original_metrics,
        "retrain": retrain_metrics,
        "unlearn": unlearn_metrics,
        "finetune": finetune_metrics,
    }

with open(PATH / "MIA_small_results", "wb") as f:
    pickle.dump(MIA_small_results, f)


In [ ]:
# TEST ON LARGE MODEL

MIA_large_results = {}

for client in UNLEARN_CLIENTS:
    retr_params = torch.load(PATH / f"{client}" / "retr_params.pt", weights_only=True)
    unl_params = torch.load(PATH / f"{client}" / "unl_params.pt", weights_only=True)
    ft_params = torch.load(PATH / f"{client}" / "ft_params.pt", weights_only=True)

    forgotten_loader = client_train_loaders[client]
    sample_size = len(forgotten_loader.dataset)
    test_non_member_loader = create_test_loader(sample_size)
    remaining_member_loader = create_remaining_loader(sample_size, client_train_idx, client)

    original_metrics = membership_inference_attack(
        og_params,
        test_non_member_loader,
        forgotten_loader,
        large_attack_model,
        DEVICE,
        focus_loader=forgotten_loader,
    )
    retrain_metrics = membership_inference_attack(
        retr_params,
        forgotten_loader,
        remaining_member_loader,
        large_attack_model,
        DEVICE,
        focus_loader=forgotten_loader,
    )
    unlearn_metrics = membership_inference_attack(
        unl_params,
        forgotten_loader,
        remaining_member_loader,
        large_attack_model,
        DEVICE,
        focus_loader=forgotten_loader,
    )
    finetune_metrics = membership_inference_attack(
        ft_params,
        forgotten_loader,
        remaining_member_loader,
        large_attack_model,
        DEVICE,
        focus_loader=forgotten_loader,
    )

    MIA_large_results[client] = {
        "original": original_metrics,
        "retrain": retrain_metrics,
        "unlearn": unlearn_metrics,
        "finetune": finetune_metrics,
    }

with open(PATH / "MIA_large_results", "wb") as f:
    pickle.dump(MIA_large_results, f)


### 5b - KL divergence and L2 distance

In [ ]:
from unlearning import predictive_kl_and_l2
KL_results = {}
L2_results = {}

for client in UNLEARN_CLIENTS:
    retr_params = torch.load(PATH / f"{client}" / "retr_params.pt", weights_only=True)
    unl_params = torch.load(PATH / f"{client}" / "unl_params.pt", weights_only=True)
    ft_params = torch.load(PATH / f"{client}" / "ft_params.pt", weights_only=True)
    
    KL_ru, L2_ru = predictive_kl_and_l2(retr_params, unl_params, test_loader, device=DEVICE)
    KL_ou, L2_ou = predictive_kl_and_l2(og_params, unl_params, test_loader, device=DEVICE)

    KL_rf, L2_rf = predictive_kl_and_l2(retr_params, ft_params, test_loader, device=DEVICE)
    KL_of, L2_of = predictive_kl_and_l2(og_params, ft_params, test_loader, device=DEVICE)

    KL_or, L2_or = predictive_kl_and_l2(og_params, retr_params, test_loader, device=DEVICE)

    print(f"\nKL(retrain || unlearn) = {KL_ru:.4f}   L2 = {L2_ru:.4f}")
    print(f"\nKL(original || unlearn) = {KL_ou:.4f}   L2 = {L2_ou:.4f}")
    print(f"\nKL(retrain || finetune) = {KL_rf:.4f}   L2 = {L2_rf:.4f}")
    print(f"\nKL(original || finetune) = {KL_of:.4f}   L2 = {L2_of:.4f}")
    print(f"\nKL(original || retrain) = {KL_or:.4f}   L2 = {L2_or:.4f}")

    KL_results[client] = {"ru": KL_ru, "ou": KL_ou, "rf": KL_rf, "of": KL_of, "or": KL_or}
    L2_results[client] = {"ru": L2_ru, "ou": L2_ou, "rf": L2_rf, "of": L2_of, "or": L2_or}

with open(PATH / "KL_results", "wb") as f:
    pickle.dump(KL_results, f)

with open(PATH / "L2_results", "wb") as f:
    pickle.dump(L2_results, f)


In [ ]:
from unlearning import predictive_metrics
from reporting import build_unlearning_pair_summary

predictive_results = {}
unlearning_summary_small = {}
unlearning_summary_large = {}

for client in UNLEARN_CLIENTS:
    client_path = PATH / f"{client}"

    retr_params = torch.load(client_path / "retr_params.pt", weights_only=True)
    unl_params = torch.load(client_path / "unl_params.pt", weights_only=True)
    ft_params = torch.load(client_path / "ft_params.pt", weights_only=True)

    with open(client_path / "retr_val_results", "rb") as f:
        retr_val_results = pickle.load(f)
    with open(client_path / "retr_test_results", "rb") as f:
        retr_test_results = pickle.load(f)
    with open(client_path / "unl_val_results", "rb") as f:
        unl_val_results = pickle.load(f)
    with open(client_path / "unl_test_results", "rb") as f:
        unl_test_results = pickle.load(f)
    with open(client_path / "ft_val_results", "rb") as f:
        ft_val_results = pickle.load(f)
    with open(client_path / "ft_test_results", "rb") as f:
        ft_test_results = pickle.load(f)

    predictive_unlearn = predictive_metrics(retr_params, unl_params, test_loader, device=DEVICE)
    predictive_original_unlearn = predictive_metrics(og_params, unl_params, test_loader, device=DEVICE)
    predictive_finetune = predictive_metrics(retr_params, ft_params, test_loader, device=DEVICE)
    predictive_original_finetune = predictive_metrics(og_params, ft_params, test_loader, device=DEVICE)
    predictive_original_retrain = predictive_metrics(og_params, retr_params, test_loader, device=DEVICE)

    predictive_results[client] = {
        "unlearn_vs_retrain": predictive_unlearn,
        "original_vs_unlearn": predictive_original_unlearn,
        "finetune_vs_retrain": predictive_finetune,
        "original_vs_finetune": predictive_original_finetune,
        "original_vs_retrain": predictive_original_retrain,
    }

    train_size = len(client_train_idx[client])
    unlearning_summary_small[client] = {
        "original_vs_retrain": build_unlearning_pair_summary(
            client,
            train_size,
            "retrain",
            "original",
            retr_val_results,
            og_val_results,
            retr_test_results,
            og_test_results,
            MIA_small_results[client]["retrain"],
            MIA_small_results[client]["original"],
            predictive_original_retrain,
        ),
        "unlearn_vs_retrain": build_unlearning_pair_summary(
            client,
            train_size,
            "retrain",
            "unlearn",
            retr_val_results,
            unl_val_results,
            retr_test_results,
            unl_test_results,
            MIA_small_results[client]["retrain"],
            MIA_small_results[client]["unlearn"],
            predictive_unlearn,
        ),
        "finetune_vs_retrain": build_unlearning_pair_summary(
            client,
            train_size,
            "retrain",
            "finetune",
            retr_val_results,
            ft_val_results,
            retr_test_results,
            ft_test_results,
            MIA_small_results[client]["retrain"],
            MIA_small_results[client]["finetune"],
            predictive_finetune,
        ),
    }
    unlearning_summary_large[client] = {
        "original_vs_retrain": build_unlearning_pair_summary(
            client,
            train_size,
            "retrain",
            "original",
            retr_val_results,
            og_val_results,
            retr_test_results,
            og_test_results,
            MIA_large_results[client]["retrain"],
            MIA_large_results[client]["original"],
            predictive_original_retrain,
        ),
        "unlearn_vs_retrain": build_unlearning_pair_summary(
            client,
            train_size,
            "retrain",
            "unlearn",
            retr_val_results,
            unl_val_results,
            retr_test_results,
            unl_test_results,
            MIA_large_results[client]["retrain"],
            MIA_large_results[client]["unlearn"],
            predictive_unlearn,
        ),
        "finetune_vs_retrain": build_unlearning_pair_summary(
            client,
            train_size,
            "retrain",
            "finetune",
            retr_val_results,
            ft_val_results,
            retr_test_results,
            ft_test_results,
            MIA_large_results[client]["retrain"],
            MIA_large_results[client]["finetune"],
            predictive_finetune,
        ),
    }

with open(PATH / "predictive_results", "wb") as f:
    pickle.dump(predictive_results, f)

with open(PATH / "unlearning_summary_small", "wb") as f:
    pickle.dump(unlearning_summary_small, f)

with open(PATH / "unlearning_summary_large", "wb") as f:
    pickle.dump(unlearning_summary_large, f)
